# ChronoReg — Qwen2.5-7B-Instruct QLoRA
Course experiment, not canonical release. Enable Internet and GPU T4. Add exact 7B ZIP as private Kaggle dataset. One GPU is used even on T4 x2. Run cells 1–3 first, inspect real GPU smoke. No 3B adapter reuse.

## 1. Locate and verify 7B dataset

In [ ]:
from pathlib import Path
import hashlib, zipfile, subprocess, sys, os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["USE_TF"] = "0"
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
os.environ.pop("BNB_CUDA_VERSION", None)
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
EXPECTED = '39d5e16f266ea05a1cdc0de96d96880af6e0ea1920b1e2202d415066209c6191'

def matching(root):
    return [x.parent for x in root.rglob('bundle_manifest.json') if hashlib.sha256(x.read_bytes()).hexdigest() == EXPECTED]

matches = matching(INPUT)
if not matches:
    dest = WORK / 'chronoreg_7b_input_fp32'
    matches = matching(dest) if dest.exists() else []
    if not matches:
        archives = list(INPUT.rglob('ChronoReg_Kaggle_7B_FP32.zip'))
        assert len(archives) == 1, 'Add the exact training ZIP as a Kaggle dataset.'
        with zipfile.ZipFile(archives[0]) as z:
            for member in z.infolist():
                assert (dest / member.filename).resolve().is_relative_to(dest.resolve()), 'Unsafe archive path'
            z.extractall(dest)
        matches = matching(dest)
assert len(matches) == 1, f'Expected one matching bundle, found {matches}'
BUNDLE = matches[0]
OUT = WORK / 'chronoreg_7b_fp32_v2'
RUNNER = BUNDLE / 'run_course_qlora.py'
def run(*args):
    subprocess.run([sys.executable, '-u', str(RUNNER), '--bundle', str(BUNDLE), '--out', str(OUT), *args], env=os.environ.copy(), check=True)
run('preflight')


## 2. Install the environment (leave Kaggle PyTorch unchanged)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(BUNDLE / 'requirements-kaggle.txt')], check=True)
run('gpu-preflight')


## 3. Real GPU smoke — stop here and send output before training
One optimizer step, actual adapter change, three validation samples, peak GPU memory. Smoke weights are not saved.

In [ ]:
run('gpu-smoke')

## 4. Train 2022 from base (only after successful smoke)

In [ ]:
run('train', '--through', '2022')

## 5. Continue remaining years (only after checking 2022)

In [ ]:
run('train', '--through', '2026')

## 6. Save checkpoints before optional generation evaluation

In [ ]:
import json, shutil
completion=json.loads((OUT/'completion.json').read_text())
assert completion['status']=='ALL_FIVE_STAGES_COMPLETE'
print(json.dumps(completion,indent=2))
archive=shutil.make_archive(str(WORK/'ChronoReg_7B_Checkpoints'),'zip',OUT)
print(archive)

## 7. Optional validation generation diagnostics (not frozen benchmark)

In [ ]:
run('evaluate')
print((OUT/'evaluation'/'summary.json').read_text())
archive=shutil.make_archive(str(WORK/'ChronoReg_7B_With_Validation'),'zip',OUT)
print(archive)